## Retrieval-Augmented Generation (RAG)

**Retrieval-Augmented Generation (RAG)** lets Claude answer questions using a large document it never saw at training time and can't fit into a single prompt. Instead of pasting the whole document into every request, we search it first and hand Claude only the relevant pieces.

The pipeline has four stages, and this notebook covers each one as its own section:

1. **Chunking** — split a large document into smaller, topically-focused pieces.
2. **Embeddings** — turn each chunk into a vector (a list of numbers) that represents its meaning.
3. **Vector DB** — store every chunk's vector so it can be searched later.
4. **Semantic Search** — embed a question and find the chunks whose meaning is closest to it, then hand those to Claude.

We use Voyage AI for embeddings (Claude itself doesn't generate embeddings) and Anthropic's Claude for the final answer. `report.md` — a 15-section fictional annual report — is the running example document.

In [ ]:
%pip install -q anthropic voyageai python-dotenv

In [ ]:
# Setup: load environment variables and create both clients
from dotenv import load_dotenv
from anthropic import Anthropic
import voyageai

load_dotenv()

anthropic_client = Anthropic()
model = "claude-sonnet-4-5"

voyage_client = voyageai.Client()

## 1. Chunking

**Definition:** Chunking is splitting a large document into smaller pieces ("chunks") before storing it for retrieval. RAG works by finding the *most relevant* pieces of a document and handing only those to Claude — but that only helps if a chunk is small enough to be about one topic. A chunk that's too big drags in irrelevant context; a chunk that's too small loses the surrounding meaning.

There are a few ways to decide where the splits go:

- **By character count** — cut every N characters, with a little overlap so a split doesn't destroy a sentence at the boundary. Simple, but can still cut mid-sentence.
- **By sentence** — group a fixed number of sentences per chunk. Keeps prose intact, but chunk sizes vary.
- **By document structure** — split on the document's own headings (e.g. every `## ` in Markdown). If the author already organized the document into topics, this gives you one chunk per topic for free — the cleanest split when the structure is available.

We'll use structure-based chunking as the running example for the rest of this notebook, since `report.md` is already organized into headed sections.

In [ ]:
def chunk_by_section(document_text):
    import re

    pattern = r"\n## "
    return re.split(pattern, document_text)

In [ ]:
with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)

print(f"{len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    preview = chunk[:80].replace("\n", " ")
    print(f"[{i}] {len(chunk):>5} chars | {preview}...")

### Other chunking strategies

For reference, here are the character-based and sentence-based strategies mentioned above. Both take a block of text and return a list of chunk strings, the same shape as `chunk_by_section` — swap them in when a document has no clear heading structure to split on.

In [ ]:
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    chunks = []
    start_idx = 0

    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))
        chunks.append(text[start_idx:end_idx])
        start_idx = end_idx - chunk_overlap if end_idx < len(text) else len(text)

    return chunks


def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    import re

    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    start_idx = 0

    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))
        chunks.append(" ".join(sentences[start_idx:end_idx]))
        start_idx = max(0, start_idx + max_sentences_per_chunk - overlap_sentences)

    return chunks

In [ ]:
char_chunks = chunk_by_char(text)
sentence_chunks = chunk_by_sentence(text)

print(f"chunk_by_char:     {len(char_chunks)} chunks")
print(f"chunk_by_sentence: {len(sentence_chunks)} chunks\n")

print("chunk_by_char[0]:")
print(repr(char_chunks[0]))

print("\nchunk_by_sentence[0]:")
print(repr(sentence_chunks[0]))

## 2. Embeddings

**Definition:** An embedding is a list of numbers (a vector) that represents the *meaning* of a piece of text. Texts with similar meaning get vectors that point in similar directions — this is what lets us search by meaning instead of by matching exact words.

We use Voyage AI's embedding model here via the `voyageai` client. `input_type` tells Voyage which side of a future search this text is on:

- `"document"` — for text we're storing (our chunks).
- `"query"` — for a question we'll search with.

The model encodes them slightly differently, and using the right one improves retrieval — we'll use both in sections 3 and 4.

In [ ]:
def generate_embedding(chunks, model="voyage-3-large", input_type="document"):
    is_list = isinstance(chunks, list)
    input = chunks if is_list else [chunks]
    result = voyage_client.embed(input, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

In [ ]:
sample_text = "Compound CTX-204b showed promising results in its Phase IIa clinical trial."

embedding = generate_embedding(sample_text)

print(f"{len(embedding)} dimensions")
print("First 5 values:", embedding[:5])

## 3. Vector DB

**Definition:** A vector database (or vector *index*, for our small-scale purposes) stores each chunk's embedding alongside the chunk itself, and answers a search by comparing a query's embedding against every stored one — returning the chunks whose meaning is closest to the query's meaning.

Here we build a minimal, dependency-free `VectorIndex`: two parallel lists (`vectors[i]` is the embedding of `documents[i]`), and a brute-force scan that scores the query against every stored vector. That's `O(n)` per search — fine for a handful of chunks. A production vector database (Pinecone, pgvector, Chroma, ...) swaps the brute-force scan for an approximate-nearest-neighbour index so it stays fast at millions of vectors, but the interface — add documents, search, get the closest matches back — is the same.

![RAG_Image](./images/rag.png)

In [ ]:
# VectorIndex implementation
#
# A minimal, dependency-free stand-in for a real vector database. It keeps two
# parallel lists - self.vectors[i] is the embedding of self.documents[i] - and
# answers a search by scoring the query against every stored vector.
import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(self, distance_metric: str = "cosine", embedding_fn=None):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError("Embedding function not provided during initialization.")
        vector = self._embedding_fn(document["content"])
        self.add_vector(vector=vector, document=document)

    def add_documents(self, documents: List[Dict[str, Any]], vectors: Optional[List[List[float]]] = None):
        # Bulk version of add_document, to avoid one API call per chunk.
        if vectors is None:
            if not self._embedding_fn:
                raise ValueError("Provide either precomputed vectors or an embedding function.")
            vectors = self._embedding_fn([doc["content"] for doc in documents])

        for vector, document in zip(vectors, documents):
            self.add_vector(vector=vector, document=document)

    def add_vector(self, vector, document: Dict[str, Any]):
        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}")

        self.vectors.append(list(vector))
        self.documents.append(document)

    def search(self, query: Any, k: int = 1) -> List[Tuple[Dict[str, Any], float]]:
        # Accepts raw text (embedded on the fly) or a ready-made vector.
        # Returns (document, distance) pairs — lower distance is a better match.
        if not self.vectors:
            return []

        if isinstance(query, str):
            query_vector = self._embedding_fn(query)
        else:
            query_vector = query

        dist_func = self._cosine_distance if self._distance_metric == "cosine" else self._euclidean_distance

        distances = [(dist_func(query_vector, v), doc) for v, doc in zip(self.vectors, self.documents)]
        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def _euclidean_distance(self, vec1, vec2) -> float:
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _cosine_distance(self, vec1, vec2) -> float:
        # 1 - cosine similarity: 0 (identical direction) to 2 (opposite).
        # Compares direction only, so a long chunk and a short chunk about the
        # same topic still score close.
        mag1 = math.sqrt(sum(x * x for x in vec1))
        mag2 = math.sqrt(sum(x * x for x in vec2))
        if mag1 == 0 or mag2 == 0:
            return 1.0
        dot = sum(p * q for p, q in zip(vec1, vec2))
        similarity = max(-1.0, min(1.0, dot / (mag1 * mag2)))
        return 1.0 - similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}')"

### Building the index

Reuse the chunks from section 1, embed every chunk with `input_type="document"`, and store the vectors alongside the chunk text.

In [ ]:
embeddings = generate_embedding(chunks, input_type="document")

store = VectorIndex()
store.add_documents([{"content": chunk} for chunk in chunks], vectors=embeddings)

store

## 4. Semantic Search

**Definition:** Semantic search is embedding a question (`input_type="query"`) and asking the `VectorIndex` for the chunks whose vectors are closest to it. Because matching happens on meaning rather than keywords, a question that shares no words with a chunk can still be its nearest vector — that's the whole point of embedding-based retrieval over plain keyword search.

We search first, then hand the retrieved chunks to Claude as context so it can answer using only the relevant part of the document — this is the full RAG loop end to end.

In [ ]:
query = "What did the Phase IIa trial of the new compound show?"
query_embedding = generate_embedding(query, input_type="query")

results = store.search(query_embedding, k=2)

for rank, (doc, distance) in enumerate(results, start=1):
    preview = doc["content"][:300].replace("\n", " ")
    print(f"--- Rank {rank} | distance {distance:.4f} ---")
    print(preview + "...\n")

### Answering with Claude

Finally, close the loop: pass the top retrieved chunk(s) to Claude as context in the prompt, and let Claude write the answer using only that retrieved text — not the whole document.

In [ ]:
context = "\n\n---\n\n".join(doc["content"] for doc, _ in results)

prompt = f"""Use the following context to answer the question. If the context doesn't contain the answer, say so.

<context>
{context}
</context>

Question: {query}"""

response = anthropic_client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[{"role": "user", "content": prompt}],
)

response.content[0].text